In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer

RANDOM_STATE = 42
N_SPLITS = 5

# ---- paths ----
TRAIN_PATHS = {
    "QB": "../data/processed/nfl_to_nfl_qb_train.csv",
    "RB": "../data/processed/nfl_to_nfl_rb_train.csv",
    "WR": "../data/processed/nfl_to_nfl_wr_train.csv",
    "TE": "../data/processed/nfl_to_nfl_te_train.csv",
}

# CFB projections file (output of cfb_to_nfl_model.py)
CFB_PROJ_PATH = "../data/processed/cfb_to_nfl_all_projections.csv"

TARGET_COL = "target_fp_ppr"

# ---- scorers ----
mse_scorer = make_scorer(mean_squared_error, greater_is_better=False)
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

# ---- Load CFB projections once, keyed by nfl_player_id ----
cfb_proj = pd.read_csv(CFB_PROJ_PATH)
cfb_proj = (
    cfb_proj[["nfl_player_id", "cfb_projected_ppr"]]
    .rename(columns={"nfl_player_id": "player_id"})
    .dropna(subset=["player_id"])
    .drop_duplicates(subset=["player_id"])  # one projection per player
)


def train_and_cv_position(pos: str, path: str):
    df = pd.read_csv(path)

    # Basic hygiene: drop rows missing target
    df = df.dropna(subset=[TARGET_COL]).copy()

    # ── Join cfb_projected_ppr (NA for veterans with no CFB data) ────────────
    df = df.merge(cfb_proj, on="player_id", how="left")

    # Define feature set: drop id + season bookkeeping + target columns
    drop_cols = {TARGET_COL, "target_games", "target_season"}
    drop_cols.update({"season"})  # comment out if you want season as a feature
    drop_cols.update({"player_id"})

    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    y = df[TARGET_COL].astype(float)

    # Coerce everything to numeric; invalid -> NaN, handled by imputer
    X = X.apply(pd.to_numeric, errors="coerce")

    model = RandomForestRegressor(
        n_estimators=600,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        min_samples_leaf=2,
    )

    pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("rf", model),
    ])

    cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    cv_results = cross_validate(
        pipe, X, y, cv=cv,
        scoring={"neg_mse": mse_scorer, "neg_mae": mae_scorer, "r2": "r2"},
        return_train_score=False,
        n_jobs=-1
    )

    summary = {
        "position": pos,
        "n_rows": len(df),
        "n_features": X.shape[1],
        "cfb_proj_coverage": f"{df['cfb_projected_ppr'].notna().sum()} / {len(df)}",
        "mse_mean": float((-cv_results["test_neg_mse"]).mean()),
        "mse_std": float((-cv_results["test_neg_mse"]).std(ddof=1)),
        "mae_mean": float((-cv_results["test_neg_mae"]).mean()),
        "mae_std": float((-cv_results["test_neg_mae"]).std(ddof=1)),
        "r2_mean": float(cv_results["test_r2"].mean()),
        "r2_std": float(cv_results["test_r2"].std(ddof=1)),
    }

    # Fit final model on all training data
    pipe.fit(X, y)

    return pipe, summary, list(X.columns)


def main():
    models = {}
    feature_cols = {}
    summaries = []

    for pos, path in TRAIN_PATHS.items():
        pipe, summary, cols = train_and_cv_position(pos, path)
        models[pos] = pipe
        feature_cols[pos] = cols
        summaries.append(summary)

    results_df = pd.DataFrame(summaries).sort_values("position")
    print("\nCross-validated performance (5-fold):")
    print(results_df.to_string(index=False))

    return models, feature_cols, results_df


if __name__ == "__main__":
    models, feature_cols, results_df = main()



Cross-validated performance (5-fold):
position  n_rows  n_features cfb_proj_coverage  mse_mean  mse_std  mae_mean  mae_std  r2_mean   r2_std
      QB     807          30         503 / 807 27.414998 3.592162  4.116327 0.208290 0.448131 0.054178
      RB    1725          34        976 / 1725 14.049023 1.889964  2.853903 0.155104 0.545242 0.046264
      TE    1303          34        721 / 1303  6.654169 0.429293  1.964139 0.105740 0.587794 0.028482
      WR    2285          34       1463 / 2285 12.094078 0.725841  2.756559 0.077874 0.590251 0.037567


In [5]:
import pickle

PREDICT_PATHS = {
    "QB": "../data/processed/nfl_to_nfl_qb_predict_2026.csv",
    "RB": "../data/processed/nfl_to_nfl_rb_predict_2026.csv",
    "WR": "../data/processed/nfl_to_nfl_wr_predict_2026.csv",
    "TE": "../data/processed/nfl_to_nfl_te_predict_2026.csv",
}

CFB_PREDICT_PATHS = {
    "QB": "../data/processed/cfb_to_nfl_qb_predict_2026.csv",
    "RB": "../data/processed/cfb_to_nfl_rb_predict_2026.csv",
    "WR": "../data/processed/cfb_to_nfl_wr_predict_2026.csv",
    "TE": "../data/processed/cfb_to_nfl_te_predict_2026.csv",
}

MODEL_DIR = "../models"

# CFB projections lookup — used to populate cfb_projected_ppr for veterans
cfb_proj = pd.read_csv("../data/processed/cfb_to_nfl_all_projections.csv")
cfb_proj = (
    cfb_proj[["nfl_player_id", "cfb_projected_ppr"]]
    .rename(columns={"nfl_player_id": "player_id"})
    .dropna(subset=["player_id"])
    .drop_duplicates(subset=["player_id"])
)


def predict_position(pos: str, model, path: str, feat_cols: list) -> pd.DataFrame:

    # ── Score veterans with the NFL model ────────────────────────────────────
    vet_df = pd.read_csv(path)
    vet_df = vet_df.merge(cfb_proj, on="player_id", how="left")

    id_cols = [c for c in ["player_id", "season"] if c in vet_df.columns]
    X_vet = vet_df.reindex(columns=feat_cols).apply(pd.to_numeric, errors="coerce")
    vet_preds = model.predict(X_vet)

    vet_out = vet_df[id_cols].copy()
    vet_out["pred_fp_ppr_2026"] = vet_preds
    vet_out["position"] = pos
    vet_out["is_rookie"] = 0
    vet_out["cfb_projected_ppr"] = vet_df["cfb_projected_ppr"].values

    # ── Score rookies with the NFL model using cfb_projected_ppr as key feature
    with open(f"{MODEL_DIR}/cfb_to_nfl_{pos.lower()}_rf.pkl", "rb") as f:
        artifact = pickle.load(f)

    cfb_df = pd.read_csv(CFB_PREDICT_PATHS[pos])
    X_cfb = pd.DataFrame(
        artifact["imputer"].transform(cfb_df.reindex(columns=artifact["feature_cols"])),
        columns=artifact["feature_cols"]
    )
    cfb_projected = artifact["model"].predict(X_cfb)

    # All NFL stats are NA for rookies — imputer fills with training medians
    # cfb_projected_ppr is the key signal the model uses for them
    rookie_X = pd.DataFrame(np.nan, index=cfb_df.index, columns=feat_cols)
    if "cfb_projected_ppr" in feat_cols:
        rookie_X["cfb_projected_ppr"] = cfb_projected
    rookie_X = rookie_X.apply(pd.to_numeric, errors="coerce")
    rookie_preds = model.predict(rookie_X)

    rookie_out = pd.DataFrame()
    rookie_out["player_id"] = cfb_df["nfl_player_id"] if "nfl_player_id" in cfb_df.columns else np.nan
    rookie_out["season"] = np.nan
    rookie_out["pred_fp_ppr_2026"] = rookie_preds
    rookie_out["position"] = pos
    rookie_out["is_rookie"] = 1
    rookie_out["cfb_projected_ppr"] = cfb_projected

    # ── Combine, sort, rank ───────────────────────────────────────────────────
    out = pd.concat([vet_out, rookie_out], ignore_index=True)
    out = out.sort_values("pred_fp_ppr_2026", ascending=False).reset_index(drop=True)
    out["rank_pos"] = np.arange(1, len(out) + 1)
    return out


# ── Generate predictions ─────────────────────────────────────────────────────
all_preds = []
for pos, path in PREDICT_PATHS.items():
    pred_df = predict_position(pos, models[pos], path, feature_cols[pos])
    all_preds.append(pred_df)
    pred_df.to_csv(f"../data/processed/nfl_to_nfl_{pos.lower()}_predictions_2026.csv", index=False)
    print(f"{pos}: {len(pred_df)} total ({(pred_df['is_rookie']==1).sum()} rookies, {(pred_df['is_rookie']==0).sum()} veterans)")

all_preds_df = pd.concat(all_preds, ignore_index=True)
all_preds_df.to_csv("../data/processed/nfl_to_nfl_all_predictions_2026.csv", index=False)
print("\nTop 5 per position:")
print(all_preds_df.groupby("position").head(5).to_string(index=False))


 player_id  season  pred_fp_ppr_2026 position  cfb_projected_ppr  rank_pos
00-0034857  2025.0         20.183545       QB          13.461365         1
00-0039851  2025.0         19.864700       QB          14.212158         2
00-0033873  2025.0         18.538625       QB          11.353202         3
00-0026498  2025.0         17.853942       QB                NaN         4
00-0036355  2025.0         17.834598       QB          14.370493         5
00-0036971  2025.0         17.639556       QB          13.501089         6
00-0036389  2025.0         17.244188       QB           6.996988         7
00-0039732  2025.0         16.833876       QB          13.725655         8
00-0033077  2025.0         16.714351       QB           7.227229         9
00-0037834  2025.0         16.674249       QB           6.667465        10
00-0033280  2025.0         22.455047       RB          14.720759         1
00-0038542  2025.0         19.060821       RB          15.176983         2
00-0039139  2025.0       

In [6]:
nfl_stats = pd.read_csv('../data/raw/nfl_player_stats_2011_2025.csv')
# 1) Build lookup from your original weekly stats (pick the right name column)
# Replace 'player_display_name' with whatever column you have.
NAME_COL = "player_display_name"  # <-- change if needed

lookup = (
    nfl_stats[["player_id", NAME_COL]]
    .dropna()
    .drop_duplicates(subset=["player_id"])
    .rename(columns={NAME_COL: "player_name"})
)

# 2) Add names to combined predictions (or do this per-position)
all_preds_df = pd.read_csv("../data/processed/nfl_to_nfl_all_predictions_2026.csv")

all_preds_named = all_preds_df.merge(lookup, on="player_id", how="left")

# (optional) reorder columns
cols = ["position", "rank_pos", "player_id", "player_name", "season", "pred_fp_ppr_2026"]
cols = [c for c in cols if c in all_preds_named.columns] + \
       [c for c in all_preds_named.columns if c not in cols]
all_preds_named = all_preds_named[cols]

all_preds_named.to_csv("../data/processed/nfl_to_nfl_all_predictions_2026_named.csv", index=False)

# quick peek
print(all_preds_named.groupby("position").head(10).to_string(index=False))

C:\Users\Samg1\AppData\Local\Temp\ipykernel_15388\4251301044.py:1: DtypeWarning: Columns (98,114) have mixed types. Specify dtype option on import or set low_memory=False.
  nfl_stats = pd.read_csv('../data/raw/nfl_player_stats_2011_2025.csv')


position  rank_pos  player_id         player_name  season  pred_fp_ppr_2026  cfb_projected_ppr
      QB         1 00-0034857          Josh Allen  2025.0         20.183545          13.461365
      QB         2 00-0039851          Drake Maye  2025.0         19.864700          14.212158
      QB         3 00-0033873     Patrick Mahomes  2025.0         18.538625          11.353202
      QB         4 00-0026498    Matthew Stafford  2025.0         17.853942                NaN
      QB         5 00-0036355      Justin Herbert  2025.0         17.834598          14.370493
      QB         6 00-0036971     Trevor Lawrence  2025.0         17.639556          13.501089
      QB         7 00-0036389         Jalen Hurts  2025.0         17.244188           6.996988
      QB         8 00-0039732              Bo Nix  2025.0         16.833876          13.725655
      QB         9 00-0033077        Dak Prescott  2025.0         16.714351           7.227229
      QB        10 00-0037834         Brock Purdy 